# DPO v1.1: 형식 단서를 제거한 선호 데이터로 재실험 (Colab T4)

v1.0에서는 chosen에만 '전원합의체·판례 변경' 문구가 있어 DPO 모델이 내용 대신 문구를 학습했다.
v1.1은 chosen/rejected를 같은 인용 형식으로 맞추고(`src/build_dpo_v11.py`), **하이퍼파라미터는 v1.0과 동일**(β=0.1, lr 5e-5, 3 epochs)하게 두어 데이터 변경 효과만 비교한다.
학습에 쓰지 않는 검증용 쌍(6쌍)의 선호 정확도도 함께 기록한다.

런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 위에서부터 차례로 실행. 전체 약 30~40분.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO_URL = "https://github.com/legal-ai-lab/Korean-Legal-IP-Reasoning-Dataset.git"
!git clone -q $REPO_URL repo
%cd repo
!pip install -q peft
!pip uninstall -y -q torchao  # Colab 기본 torchao(구버전)가 peft와 충돌

## 1) v1.1 선호 데이터 다시 만들기 (몇 초)
단서 문구가 남아 있으면 여기서 실패한다.

In [ ]:
!python src/build_dpo_v11.py

## 2) SFT 다시 학습 (약 5분)
v1.0 세션의 SFT 어댑터는 남아 있지 않아서 같은 설정으로 다시 학습한다.

In [ ]:
!python src/train_sft.py --epochs 3

## 3) 재학습한 SFT 평가 (약 5~8분)
DPO v1.1과 같은 세션의 SFT를 기준으로 비교하기 위해 `sft_run2`로 저장한다.

In [ ]:
!python src/evaluate.py --name sft_run2 --adapter outputs/qwen2.5-1.5b-legal-lora

## 4) DPO v1.1 학습 (약 3~5분)
출력 마지막의 `val pref acc`가 학습에 쓰지 않은 쌍의 선호 정확도다.

In [ ]:
!python src/train_dpo.py --data data/legal_dpo_dataset_v11.jsonl \
  --out outputs/qwen2.5-1.5b-legal-dpo-v11-lora --log results/dpo_v11_train_log.json

## 5) DPO v1.1 평가 · 비교표 · 인용 실재성 검사 (약 5~8분)

In [ ]:
!python src/evaluate.py --name dpo_v11 --merge_adapter outputs/qwen2.5-1.5b-legal-lora --adapter outputs/qwen2.5-1.5b-legal-dpo-v11-lora
!python src/evaluate.py --report base sft dpo sft_run2 dpo_v11
!python src/evaluate.py --citations base sft dpo sft_run2 dpo_v11

## 6) 결과 내려받기
`results_v11.zip`을 Claude에게 올리면 된다.

In [ ]:
!zip -qr results_v11.zip results data/dpo_v11_report.json
from google.colab import files
files.download("results_v11.zip")